### Imports and warning handling


In [47]:
import os
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

warnings.filterwarnings("ignore", category=ConvergenceWarning)

### Locate the input data


In [48]:
# Locate the Week 5 input datasets without embedding personal file paths.
# Module 16/Data is preferred; the previous week's saved data is a fallback.
required = [f"function_{i}.csv" for i in range(1, 9)]
roots = [Path.cwd(), *list(Path.cwd().parents)]
relative_folders = [
    Path("Data"),
    Path("5. Module 16") / "Data",
    Path("7. Capstone Project") / "5. Module 16" / "Data",
    Path("4. Module 15") / "updated_data_week_4",
    Path("7. Capstone Project") / "4. Module 15" / "updated_data_week_4",
    Path("updated_data_week_4"),
]
possible = [root / rel for root in roots for rel in relative_folders]
load_folder = next((p for p in possible if all((p / name).is_file() for name in required)), None)
if load_folder is None:
    raise FileNotFoundError(
        "Eight function CSVs were not found. Start Jupyter from the capstone project "
        "folder, or place function_1.csv through function_8.csv in Module 16/Data."
    )
print("Input datasets located successfully.")

Input datasets located successfully.


In [49]:
data = {}
for i in range(1, 9):
    data[i] = pd.read_csv(load_folder / f'function_{i}.csv')

print("Eight datasets loaded:", ", ".join(f"F{i}: {len(data[i])} rows" for i in range(1, 9)))

Eight datasets loaded: F1: 14 rows, F2: 14 rows, F3: 19 rows, F4: 34 rows, F5: 24 rows, F6: 24 rows, F7: 34 rows, F8: 44 rows


### Add the returned observations


In [50]:
new_inputs = {
    1: [0.417430, 0.463246],
    2: [0.730707, 0.915532],
    3: [0.433866, 0.716516, 0.000863],
    4: [0.443740, 0.493644, 0.409790, 0.452219],
    5: [0.851064, 0.985689, 0.919598, 0.979330],
    6: [0.033216, 0.058439, 0.124012, 0.011504, 0.042834],
    7: [0.154752, 0.313081, 0.341855, 0.169387, 0.318877, 0.633022],
    8: [0.148540, 0.297539, 0.201860, 0.328776,
        0.636564, 0.500531, 0.197949, 0.479426]
}

new_outputs = {
    1: -0.005912680468954885,
    2: 0.5074182645857468,
    3: -0.10720630327643242,
    4: -1.4386431207074888,
    5: 4857.449601416715,
    6: -2.043879876538988,
    7: 2.536938996958511,
    8: 9.9112493761444
}

In [51]:
previous_bests = {
    i: data[i]["output"].max()
    for i in range(1, 9)
}

In [52]:
for i in range(1, 9):
    feature_columns = [col for col in data[i].columns if col != 'output']
    point = np.asarray(new_inputs[i], dtype=float)
    X_existing = data[i][feature_columns].to_numpy(dtype=float)
    matches = np.all(np.isclose(X_existing, point, rtol=0, atol=1e-08), axis=1)
    if matches.any():
        observed = data[i].loc[matches, 'output'].to_numpy(dtype=float)
        if not np.any(np.isclose(observed, new_outputs[i], rtol=1e-08, atol=1e-08)):
            raise ValueError(f'Function {i}: existing point has a different output; check the input dataset.')

    else:
        new_row = dict(zip(feature_columns, point))
        new_row['output'] = new_outputs[i]
        data[i] = pd.concat([data[i], pd.DataFrame([new_row])], ignore_index=True)

print("Latest observations checked; existing rows were not duplicated.")

Latest observations checked; existing rows were not duplicated.


### Step 1 — Evaluate the latest outputs


In [53]:
for i in range(1, 9):
    previous_best = previous_bests[i]
    new_output = new_outputs[i]
    updated_best = data[i]['output'].max()
    difference = new_output - previous_best

display(pd.DataFrame([{"Function":i,"Previous best":previous_bests[i],"New output":new_outputs[i],"Updated best":data[i]["output"].max(),"Improved":new_outputs[i]>previous_bests[i]} for i in range(1,9)]).round(6).set_index("Function"))

,Previous best,New output,Updated best,Improved
Function,,,,
1,0.000000,-0.005913,0.000000,False
2,0.611205,0.507418,0.611205,False
3,-0.020167,-0.107206,-0.020167,False
4,-0.300645,-1.438643,-0.300645,False
5,4857.449601,4857.449601,4857.449601,False
6,-0.458511,-2.043880,-0.458511,False
7,2.536939,2.536939,2.536939,False
8,9.911249,9.911249,9.911249,False


### Step 2 — Refit Gaussian Processes


In [54]:
models_week5 = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values
    y = data[i]['output'].values
    kernel = RBF(length_scale=1.0, length_scale_bounds=(1e-05, 100000.0))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=10, random_state=42)
    gp.fit(X, y)
    models_week5[i] = gp

display(pd.DataFrame([{"Function":i,"GP kernel":str(models_week5[i].kernel_)} for i in range(1,9)]).set_index("Function"))

,GP kernel
Function,
1,RBF(length_scale=1e-05)
2,RBF(length_scale=0.0494)
3,RBF(length_scale=0.173)
4,RBF(length_scale=0.414)
5,RBF(length_scale=0.467)
6,RBF(length_scale=1e-05)
7,RBF(length_scale=0.33)
8,RBF(length_scale=0.979)


### GP prediction diagnostics


In [55]:
rng = np.random.default_rng(42)
gp_diagnostics = {}
for i in range(1, 9):
    n_dimensions = len(data[i].columns) - 1
    candidates = rng.random((100000, n_dimensions))
    mean_pred, std_pred = models_week5[i].predict(candidates, return_std=True)
    mean_range = mean_pred.max() - mean_pred.min()
    std_range = std_pred.max() - std_pred.min()
    gp_diagnostics[i] = {'mean_min': mean_pred.min(), 'mean_max': mean_pred.max(), 'mean_range': mean_range, 'std_min': std_pred.min(), 'std_max': std_pred.max(), 'std_range': std_range}

display(pd.DataFrame([{"Function":i,"Mean range":gp_diagnostics[i]["mean_range"],"Uncertainty range":gp_diagnostics[i]["std_range"]} for i in range(1,9)]).round(6).set_index("Function"))

,Mean range,Uncertainty range
Function,,
1,0.000680,0.001703
2,0.699132,0.237428
3,0.389118,0.077909
4,34.460542,7.316418
5,6264.230066,1228.812886
6,1.149715,0.509894
7,2.487448,0.642344
8,4.801047,0.669037


### Step 3 — Acquisition functions


In [56]:
from scipy.stats import norm

def expected_improvement(mean, std, best_y, xi=0.01):
    improvement = mean - best_y - xi

    with np.errstate(divide="ignore", invalid="ignore"):
        Z = improvement / std
        ei = (
            improvement * norm.cdf(Z)
            + std * norm.pdf(Z)
        )

    ei[std == 0] = 0
    return ei

### Compare UCB and Expected Improvement


In [57]:
kappa = 1.5
xi = 0.01
n_candidates = 100000
gp_candidate_results = {}
for i in range(2, 9):
    X_existing = data[i].drop(columns='output').values
    y_existing = data[i]['output'].values
    n_dimensions = X_existing.shape[1]
    rng = np.random.default_rng(100 + i)
    X_candidates = rng.uniform(0, 1, size=(n_candidates, n_dimensions))
    mean, std = models_week5[i].predict(X_candidates, return_std=True)
    best_index = np.argmax(y_existing)
    best_y = y_existing[best_index]
    best_point = X_existing[best_index]
    ucb = mean + kappa * std
    ucb_index = np.argmax(ucb)
    ucb_point = X_candidates[ucb_index]
    ei = expected_improvement(mean, std, best_y, xi=xi)
    ei_index = np.argmax(ei)
    ei_point = X_candidates[ei_index]
    ucb_distance_best = np.linalg.norm(ucb_point - best_point)
    ei_distance_best = np.linalg.norm(ei_point - best_point)
    ucb_nearest = np.linalg.norm(X_existing - ucb_point, axis=1).min()
    ei_nearest = np.linalg.norm(X_existing - ei_point, axis=1).min()
    distance_between = np.linalg.norm(ucb_point - ei_point)
    gp_candidate_results[i] = {'ucb_point': ucb_point, 'ucb_mean': mean[ucb_index], 'ucb_std': std[ucb_index], 'ucb_score': ucb[ucb_index], 'ucb_distance_best': ucb_distance_best, 'ucb_nearest': ucb_nearest, 'ei_point': ei_point, 'ei_mean': mean[ei_index], 'ei_std': std[ei_index], 'ei_score': ei[ei_index], 'ei_distance_best': ei_distance_best, 'ei_nearest': ei_nearest, 'distance_between': distance_between}

gp_rows = []
for i in range(2, 9):
    r = gp_candidate_results[i]
    gp_rows.extend([
        {"Function": i, "Method": "UCB (κ=1.5)", "Predicted output": r["ucb_mean"],
         "Uncertainty": r["ucb_std"], "Acquisition score": r["ucb_score"],
         "Distance to best": r["ucb_distance_best"], "Nearest observation": r["ucb_nearest"]},
        {"Function": i, "Method": "EI", "Predicted output": r["ei_mean"],
         "Uncertainty": r["ei_std"], "Acquisition score": r["ei_score"],
         "Distance to best": r["ei_distance_best"], "Nearest observation": r["ei_nearest"]},
    ])
display(pd.DataFrame(gp_rows).round(5).set_index(["Function", "Method"]))
display(pd.DataFrame([{"Function": i, "UCB–EI distance": gp_candidate_results[i]["distance_between"]}
                      for i in range(2, 9)]).round(5).set_index("Function"))

Predicted output  Uncertainty  Acquisition score  \
Function Method                                                          
2        UCB (κ=1.5)           0.43187      0.18715            0.71260   
         EI                    0.42944      0.18875            0.01525   
3        UCB (κ=1.5)          -0.03499      0.05509            0.04764   
         EI                   -0.01324      0.03605            0.01290   
4        UCB (κ=1.5)          -0.36561      0.82247            0.86810   
         EI                   -0.21253      0.70844            0.32340   
5        UCB (κ=1.5)        4895.38264    318.69528         5373.42556   
         EI                 5007.17175    244.10920          190.00349   
6        UCB (κ=1.5)          -1.51061      0.53464           -0.70865   
         EI                   -1.51061      0.53464            0.00471   
7        UCB (κ=1.5)           2.42715      0.23766            2.78363   
         EI                    2.42715      0.23766            0.04671   
8        UCB (κ=1.5)           9.63244      0.44318           10.29721   
         EI                    9.91540      0.22065            0.08513   

                      Distance to best  Nearest observation  
Function Method                                              
2        UCB (κ=1.5)           0.81376              0.04866  
         EI                    0.78848              0.04937  
3        UCB (κ=1.5)           0.21023              0.18318  
         EI                    0.13246              0.13246  
4        UCB (κ=1.5)           0.13029              0.13029  
         EI                    0.11417              0.11417  
5        UCB (κ=1.5)           0.14681              0.14681  
         EI                    0.10960              0.10960  
6        UCB (κ=1.5)           1.30996              0.74515  
         EI                    1.30996              0.74515  
7        UCB (κ=1.5)           0.14008              0.14008  
         EI                    0.14008              0.14008  
8        UCB (κ=1.5)           0.78319              0.72986  
         EI                    0.53771              0.48958

,UCB–EI distance
Function,
2,0.02528
3,0.16113
4,0.01926
5,0.04437
6,0.00000
7,0.00000
8,0.73196


### Step 4 — Neural-network imports


In [58]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.preprocessing import StandardScaler

### Define the neural network


In [59]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)

### Train the neural-network ensemble


In [60]:
nn_models = {}
nn_scalers = {}
nn_training_losses = {}
n_networks = 5
n_epochs = 1000
learning_rate = 0.01
for i in range(1, 9):
    X = data[i].drop(columns='output').values
    y = data[i]['output'].values.reshape(-1, 1)
    X_scaler = StandardScaler()
    y_scaler = StandardScaler()
    X_scaled = X_scaler.fit_transform(X)
    y_scaled = y_scaler.fit_transform(y)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    y_tensor = torch.tensor(y_scaled, dtype=torch.float32)
    function_models = []
    function_losses = []
    for seed in range(n_networks):
        torch.manual_seed(seed)
        model = SimpleNN(input_dim=X.shape[1])
        optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)
        loss_function = nn.MSELoss()
        for epoch in range(n_epochs):
            optimiser.zero_grad()
            predictions = model(X_tensor)
            loss = loss_function(predictions, y_tensor)
            loss.backward()
            optimiser.step()
        function_models.append(model)
        function_losses.append(loss.item())
    nn_models[i] = function_models
    nn_scalers[i] = {'X': X_scaler, 'y': y_scaler}
    nn_training_losses[i] = function_losses

display(pd.DataFrame([{"Function":i,"Mean training loss":np.mean(nn_training_losses[i])} for i in range(1,9)]).round(6).set_index("Function"))

,Mean training loss
Function,
1,0.001044
2,0.002267
3,0.000001
4,0.000000
5,0.000004
6,0.000013
7,0.000000
8,0.000000


### Leave-one-out cross-validation


In [61]:
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error
nn_cv_results = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values
    y = data[i]['output'].values
    loo = LeaveOneOut()
    actual_values = []
    nn_predictions = []
    baseline_predictions = []
    for fold, (train_index, test_index) in enumerate(loo.split(X)):
        X_train = X[train_index]
        X_test = X[test_index]
        y_train = y[train_index]
        y_test = y[test_index]
        X_scaler = StandardScaler()
        y_scaler = StandardScaler()
        X_train_scaled = X_scaler.fit_transform(X_train)
        X_test_scaled = X_scaler.transform(X_test)
        y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1))
        X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
        X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
        torch.manual_seed(42 + fold)
        model = SimpleNN(input_dim=X.shape[1])
        optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)
        loss_function = nn.MSELoss()
        for epoch in range(n_epochs):
            optimiser.zero_grad()
            predictions = model(X_train_tensor)
            loss = loss_function(predictions, y_train_tensor)
            loss.backward()
            optimiser.step()
        model.eval()
        with torch.no_grad():
            prediction_scaled = model(X_test_tensor).numpy().reshape(-1, 1)
        prediction = y_scaler.inverse_transform(prediction_scaled)[0, 0]
        actual_values.append(y_test[0])
        nn_predictions.append(prediction)
        baseline_predictions.append(np.mean(y_train))
    nn_mae = mean_absolute_error(actual_values, nn_predictions)
    baseline_mae = mean_absolute_error(actual_values, baseline_predictions)
    nn_cv_results[i] = {'nn_mae': nn_mae, 'baseline_mae': baseline_mae}

display(pd.DataFrame([{"Function":i,"NN LOOCV MAE":nn_cv_results[i]["nn_mae"],"Baseline MAE":nn_cv_results[i]["baseline_mae"],"NN better":nn_cv_results[i]["nn_mae"]<nn_cv_results[i]["baseline_mae"]} for i in range(1,9)]).round(6).set_index("Function"))

,NN LOOCV MAE,Baseline MAE,NN better
Function,,,
1,0.002595,0.001255,False
2,0.188740,0.225146,True
3,0.066222,0.046560,False
4,1.998674,6.651296,True
5,192.581456,1041.351340,True
6,0.327649,0.453136,True
7,0.315686,0.474058,True
8,0.328345,0.900332,True


### Generate neural-network candidates


In [62]:
nn_candidate_results = {}
n_candidates = 100000
for i in range(1, 9):
    X_existing = data[i].drop(columns='output').values
    n_dimensions = X_existing.shape[1]
    rng = np.random.default_rng(200 + i)
    X_candidates = rng.uniform(0, 1, size=(n_candidates, n_dimensions))
    X_scaler = nn_scalers[i]['X']
    y_scaler = nn_scalers[i]['y']
    X_candidates_scaled = X_scaler.transform(X_candidates)
    X_tensor = torch.tensor(X_candidates_scaled, dtype=torch.float32)
    network_predictions = []
    for model in nn_models[i]:
        model.eval()
        with torch.no_grad():
            pred_scaled = model(X_tensor).numpy().reshape(-1, 1)
        pred_original = y_scaler.inverse_transform(pred_scaled).ravel()
        network_predictions.append(pred_original)
    network_predictions = np.array(network_predictions)
    mean_prediction = network_predictions.mean(axis=0)
    std_prediction = network_predictions.std(axis=0)
    best_index = np.argmax(mean_prediction)
    selected_point = X_candidates[best_index]
    selected_mean = mean_prediction[best_index]
    selected_std = std_prediction[best_index]
    distances_existing = np.linalg.norm(X_existing - selected_point, axis=1)
    nearest_distance = distances_existing.min()
    current_best_index = data[i]['output'].idxmax()
    current_best_point = data[i].drop(columns='output').loc[current_best_index].values.astype(float)
    distance_best = np.linalg.norm(selected_point - current_best_point)
    nn_candidate_results[i] = {'point': selected_point, 'predicted_mean': selected_mean, 'prediction_std': selected_std, 'nearest_distance': nearest_distance, 'distance_best': distance_best}

display(pd.DataFrame([
    {"Function": i, "NN prediction": nn_candidate_results[i]["predicted_mean"],
     "Model disagreement": nn_candidate_results[i]["prediction_std"],
     "Distance to best": nn_candidate_results[i]["distance_best"],
     "Nearest observation": nn_candidate_results[i]["nearest_distance"]}
    for i in range(1, 9)
]).round(5).set_index("Function"))

,NN prediction,Model disagreement,Distance to best,Nearest observation
Function,,,,
1,0.004560,0.001420,0.77317,0.29824
2,1.116520,0.831740,0.97216,0.35486
3,0.000980,0.027670,0.52621,0.34770
4,-0.570390,0.191090,0.05778,0.05778
5,5001.947266,90.387283,0.08830,0.08830
6,-0.403260,0.039520,0.06589,0.06589
7,2.835740,0.333870,0.35448,0.35448
8,10.356460,0.134740,0.38431,0.38431


### Step 5 — Compare GP and NN candidates


In [63]:
i = 1
kappa = 1.5
xi = 0.01
n_candidates = 100000
X_existing = data[i].drop(columns='output').values
y_existing = data[i]['output'].values
n_dimensions = X_existing.shape[1]
rng = np.random.default_rng(100 + i)
X_candidates = rng.uniform(0, 1, size=(n_candidates, n_dimensions))
mean, std = models_week5[i].predict(X_candidates, return_std=True)
best_index = np.argmax(y_existing)
best_y = y_existing[best_index]
best_point = X_existing[best_index]
ucb = mean + kappa * std
ucb_index = np.argmax(ucb)
ucb_point = X_candidates[ucb_index]
ei = expected_improvement(mean, std, best_y, xi=xi)
ei_index = np.argmax(ei)
ei_point = X_candidates[ei_index]
ucb_distance_best = np.linalg.norm(ucb_point - best_point)
ei_distance_best = np.linalg.norm(ei_point - best_point)
ucb_nearest = np.linalg.norm(X_existing - ucb_point, axis=1).min()
ei_nearest = np.linalg.norm(X_existing - ei_point, axis=1).min()
distance_between = np.linalg.norm(ucb_point - ei_point)
gp_candidate_results[i] = {'ucb_point': ucb_point, 'ucb_mean': mean[ucb_index], 'ucb_std': std[ucb_index], 'ucb_score': ucb[ucb_index], 'ucb_distance_best': ucb_distance_best, 'ucb_nearest': ucb_nearest, 'ei_point': ei_point, 'ei_mean': mean[ei_index], 'ei_std': std[ei_index], 'ei_score': ei[ei_index], 'ei_distance_best': ei_distance_best, 'ei_nearest': ei_nearest, 'distance_between': distance_between}

print("Function 1 GP candidates generated for model comparison.")

Function 1 GP candidates generated for model comparison.


### Compare model recommendations


In [64]:
model_comparison = {}
for i in range(1, 9):
    nn_point = nn_candidate_results[i]['point']
    ucb_point = gp_candidate_results[i]['ucb_point']
    ei_point = gp_candidate_results[i]['ei_point']
    nn_to_ucb = np.linalg.norm(nn_point - ucb_point)
    nn_to_ei = np.linalg.norm(nn_point - ei_point)
    ucb_to_ei = np.linalg.norm(ucb_point - ei_point)
    model_comparison[i] = {'nn_to_ucb': nn_to_ucb, 'nn_to_ei': nn_to_ei, 'ucb_to_ei': ucb_to_ei}

display(pd.DataFrame([{"Function":i,"NN–UCB":model_comparison[i]["nn_to_ucb"],"NN–EI":model_comparison[i]["nn_to_ei"],"UCB–EI":model_comparison[i]["ucb_to_ei"]} for i in range(1,9)]).round(4).set_index("Function"))

,NN–UCB,NN–EI,UCB–EI
Function,,,
1,0.3605,0.3605,0.0000
2,0.3066,0.3169,0.0253
3,0.7054,0.6475,0.1611
4,0.1381,0.1198,0.0193
5,0.0966,0.0718,0.0444
6,1.3150,1.3150,0.0000
7,0.2738,0.2738,0.0000
8,0.7429,0.7940,0.7320


### Step 6 — Neighbourhood evidence


In [65]:
for i in range(1, 9):
    X_existing = data[i].drop(columns='output').values
    y_existing = data[i]['output'].values
    candidates = {'UCB': gp_candidate_results[i]['ucb_point'], 'EI': gp_candidate_results[i]['ei_point'], 'NN': nn_candidate_results[i]['point']}

    for method, candidate in candidates.items():
        distances = np.linalg.norm(X_existing - candidate, axis=1)
        nearest_indices = np.argsort(distances)[:3]

print("Nearest-observation evidence calculated for all three methods.")

Nearest-observation evidence calculated for all three methods.


### Step 7 — Final selection and validation


In [66]:
final_selections = {1: {'point': gp_candidate_results[1]['ei_point'], 'method': 'EI / UCB agreement'}, 2: {'point': gp_candidate_results[2]['ei_point'], 'method': 'Expected Improvement'}, 3: {'point': gp_candidate_results[3]['ei_point'], 'method': 'Expected Improvement'}, 4: {'point': gp_candidate_results[4]['ei_point'], 'method': 'Expected Improvement'}, 5: {'point': gp_candidate_results[5]['ei_point'], 'method': 'Expected Improvement'}, 6: {'point': nn_candidate_results[6]['point'], 'method': 'Neural Network'}, 7: {'point': gp_candidate_results[7]['ei_point'], 'method': 'EI / UCB agreement'}, 8: {'point': nn_candidate_results[8]['point'], 'method': 'Neural Network'}}

all_valid = True
for i in range(1, 9):
    point = np.asarray(final_selections[i]['point'], dtype=float)
    method = final_selections[i]['method']
    X_existing = data[i].drop(columns='output').values.astype(float)
    in_bounds = np.all((point >= 0) & (point <= 1))
    distances = np.linalg.norm(X_existing - point, axis=1)
    nearest_distance = distances.min()
    duplicate = np.any(np.all(np.isclose(X_existing, point, atol=1e-08), axis=1))
    valid = in_bounds and (not duplicate)
    if not valid:
        all_valid = False

display(pd.DataFrame([{"Function":i,"Method":final_selections[i]["method"],"Nearest distance":float(np.linalg.norm(data[i].drop(columns="output").values-final_selections[i]["point"],axis=1).min()),"In bounds":bool(np.all((np.asarray(final_selections[i]["point"])>=0)&(np.asarray(final_selections[i]["point"])<=1)))} for i in range(1,9)]).round(6).set_index("Function"))
print("All eight points valid:", all_valid)

,Method,Nearest distance,In bounds
Function,,,
1,EI / UCB agreement,0.140043,True
2,Expected Improvement,0.049374,True
3,Expected Improvement,0.132463,True
4,Expected Improvement,0.114172,True
5,Expected Improvement,0.109600,True
6,Neural Network,0.065893,True
7,EI / UCB agreement,0.140085,True
8,Neural Network,0.384306,True


All eight points valid: True


### Submission coordinates


In [67]:
print("WEEK 5 SUBMISSION POINTS")
print("=" * 70)

for i in range(1, 9):

    point = np.asarray(
        final_selections[i]["point"],
        dtype=float
    )

    formatted = "-".join(
        f"{x:.6f}" for x in point
    )

    print(f"Function {i}: {formatted}")

WEEK 5 SUBMISSION POINTS
Function 1: 0.943533-0.359421
Function 2: 0.713092-0.138155
Function 3: 0.384686-0.478261-0.416295
Function 4: 0.462451-0.342242-0.380323-0.445636
Function 5: 0.938942-0.989452-0.984552-0.971823
Function 6: 0.248909-0.369411-0.484746-0.674732-0.011701
Function 7: 0.132842-0.294043-0.285122-0.050957-0.281982-0.619768
Function 8: 0.034135-0.275291-0.032179-0.508682-0.760754-0.617950-0.034127-0.349420
